In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
from load_config_dataset import *

train_config, simulator_config, simulator_ranges, simulator_ranges_training, simulator_ranges_testing = load_all_configs(config_dir="./configs")
train_config["device"] = "cuda:0"
train_config["gpu"] = 0

dataset = load_dataset(train_config)


In [ ]:
from setup_model_params import *
cfg, models, coordinates_visual_field = build_models(train_config=train_config, simulator_config=simulator_config,simulator_ranges=simulator_ranges)
check_point = load_checkpoint(models, "xxx.ckpt", cfg["device"])

In [ ]:
print(cfg["device"])

## Define patient (simulator parameters and mapping)

In [ ]:
print(torch.initial_seed())

In [ ]:

import os
import uuid
import numpy as np
import virtual_patient
import visualization
import importlib
importlib.reload(virtual_patient)
importlib.reload(visualization)
from visualization import *
from virtual_patient import *
from pathlib import Path


fixed_patient = True  # Keep the patient fixed
use_VLM = False  # Use VLM for recognition

## Modifications ===============================
noise_level = 10  # Noise level to add, range=[0~100]  ## manual adjustment
run_id = "xxx"  ## temporary subject/patient ID
user_name = 'xxx'                            ## subject name
## =======================================
patient_dir = Path(f"results/noise_exp_{run_id}")
os.makedirs(patient_dir, exist_ok=True)

if fixed_patient:

    patient_file = patient_dir / "patient_state.npy"
    if not patient_file.exists():
        raise FileNotFoundError(f"Patient state file not found: {patient_file}")
    patient_state_ = np.load(patient_file, allow_pickle=True).item()
    print(f"Loading patient from: {patient_file}")

    sampled_params = patient_state_['sampled_params']
    non_linear_num = patient_state_['non_linear_num']

    affine_params = torch.as_tensor(
        patient_state_['affine_params'],
        dtype=torch.float32,
        device=cfg["device"],
    ).view(1, 7)

    local_params = torch.as_tensor(
        patient_state_['local_params'],
        dtype=torch.float32,
        device=cfg["device"],
    )

    theta = torch.as_tensor(
        patient_state_['theta'],
        dtype=torch.float32,
        device=cfg["device"],
    )

    patient = VirtualPatient(
        sampled_params=sampled_params,
        simulator_ranges=simulator_ranges,
        simulator_config=simulator_config,
        coordinates_visual_field=coordinates_visual_field,
        cfg=cfg,
        affine_params=affine_params,
        local_params=local_params,
        non_linear_num=non_linear_num,
        theta=theta,
        use_VLM=use_VLM,
        patient_id=run_id,
    )

else:
    
    run_id = uuid.uuid4().hex[:8]
    patient_dir = Path(f"results/noise_exp_{run_id}")
    os.makedirs(patient_dir, exist_ok=True)
    print(f"Creating new patient with ID: {run_id}")
    
    cfg["results_dir"] = patient_dir
    print(f"Results dir: {patient_dir}")

    sampled_params = sample_simulator_params(simulator_ranges, train_config=train_config,seed=None) # If using default simulator_config, sampled_params=None
  
    coordinates_visual_field = pickle.load(open(cfg['phosphene_map'], 'rb'))
    affine_params = torch.zeros(1, 7)  # Use zero affine transform parameters

    patient = VirtualPatient(
        sampled_params=sampled_params, 
        simulator_ranges=simulator_ranges,
        simulator_config=simulator_config,
        coordinates_visual_field=coordinates_visual_field,
        cfg=cfg,
        affine_params=None,
        local_params = None,
        non_linear_num=10,
        theta=None,
        use_VLM=use_VLM,
        patient_id=run_id,
        )
    print(patient.affine_params)

    patient_state = {
        'patient_id': patient.patient_id,
        'sampled_params': patient.sampled_params,
        'simulator_ranges': patient.simulator_ranges,
        'affine_params': patient.affine_params.cpu().numpy(),
        'local_params': patient.local_params.cpu().numpy(),
        'non_linear_num': patient.non_linear_num,
        'theta': patient.theta.cpu().numpy()
    }

    np.save(patient_dir / "patient_state.npy", patient_state, allow_pickle=True)


In [ ]:
print(f"Loading patient from: {patient_file}")

results_dir = patient_dir / f"noise_{noise_level}"/ f"user_{user_name}"
os.makedirs(results_dir, exist_ok=True)
print(f"Results dir: {results_dir}")

print(f"Use VLM: {use_VLM}")
print(f"Fixed Patient: {fixed_patient}")
print(f"noise level: {noise_level}") 

In [ ]:
import visualization
importlib.reload(visualization)
from visualization import *
outputs, sample_indices, figs, axes, eval_answers = visualize_demo_samples_multi_patients(patients=patient,dataset=dataset,dataset_type="dino_set",models=models,deterministic=True, cfg=cfg,fig_num=1,)
outputs, sample_indices, figs, axes, eval_answers = visualize_demo_samples_multi_patients(patients=patient,dataset=dataset,dataset_type="calibration_set",models=models,deterministic=True, cfg=cfg,fig_num=4,)

## DPO definition (reference policy, policy, optimizer)

In [ ]:
import optimization
importlib.reload(optimization)
from optimization import *

policy = models['encoder']
ref_policy = copy.deepcopy(policy).eval()

for p in ref_policy.parameters():
    p.requires_grad = False

for p in policy.parameters():
    p.requires_grad = False

base_encoder = policy.base_encoder
shift_params = [base_encoder.global_shift_mean, base_encoder.global_shift_log_std]
rotate_params = [base_encoder.global_rotate_mean,base_encoder.global_rotate_log_std]

rbf_params = [base_encoder.rbf_shift_mean, base_encoder.rbf_shift_log_std, 
              base_encoder.rbf_gate_logit_mean, base_encoder.rbf_gate_logit_log_std,
              base_encoder.rbf_raw_sigma_mean,base_encoder.rbf_raw_sigma_log_std]

brightness_params = [base_encoder.stim_scale_raw_mean, base_encoder.stim_scale_raw_log_std]
all_param_groups = [shift_params,rotate_params,rbf_params,brightness_params]

optimizer_shift = make_adamw(shift_params, lr=1e-1, betas=(0.9, 0.999))
optimizer_rotate = make_adamw(rotate_params, lr=1e-1, betas=(0.9, 0.999))
optimizer_rbf = make_adamw(rbf_params, lr=1e-1, betas=(0.9, 0.999))
optimizer_brightness = make_adamw(brightness_params, lr=1e-1, betas=(0.9, 0.999) )


## DPO

In [ ]:
policy.train()

beta = 1
K = 2
batch_size = 2
noise_prob = noise_level / 100.0
total_steps =100

# Whether to resume training
resume_training = False

if resume_training:
    total_steps =100
    resume_ckpt_path = os.path.join(results_dir, "dpo_checkpoint_step_0030.pt")

    resume = load_dpo_checkpoint(
        path=resume_ckpt_path,
        policy=policy,
        optimizer_shift=optimizer_shift,
        optimizer_rotate=optimizer_rotate,
        optimizer_brightness=optimizer_brightness,
        device=cfg["device"],
        restore_rng=True,
    )

    start_step = resume["start_step"]
    logs = resume["logs"]
    generator = resume["generator"]
    noise_rng = resume["noise_rng"]

    print(f"Resumed from {resume_ckpt_path}")
    print(f"start_step = {start_step}")

else:
    start_step = 0

    # Fix RBF exploration scale manually only for new training
    base_encoder.rbf_gate_logit_log_std.data.fill_(-20)
    base_encoder.rbf_raw_sigma_log_std.data.fill_(-20)
    base_encoder.rbf_shift_log_std.data.fill_(-20)

    generator = torch.Generator().manual_seed(0)
    noise_rng = random.Random(123)

    logs = {
        "config": {
            "noise_level": noise_level,
            "noise_seed": 123,
            "batch_size": batch_size,
            "K": K,
            "beta": beta,
        },
        "steps": [],
    }

    torch.save(ref_policy.state_dict(), os.path.join(results_dir, f"ref_policy.pt"))
    print("Start new DPO training")


In [ ]:
import numpy as np
import random
import glob

for step in range(start_step, total_steps):

    print(f"========================Step {step}============================")

    x, sample_indices = get_manual_batch(dataset=dataset, dataset_type="calibration_set", batch_size=2, device=cfg['device'], mode='random', fix_idx=4, generator=generator) # [B,C,H,W]
    # break
    # 1) Sample K actions from the current policy
    K = 2
    x_transformed = patient.transform_image(x)
    actions, stimulations, _, _, _ = policy.sample_n_actions(x_transformed, K)

    shift_rank = []
    rotate_rank = []
    rbf_rank = []
    brightness_rank = []

    human_shift_rank = []
    human_rotate_rank = []
    human_brightness_rank = []

    noise_events = {"shift": [], "rotate": [], "brightness": []}
    label_inputs = []

    for b in range(x.shape[0]):
        stim_b = stimulations[:, b, :]   # [K, E]

        simulator = patient.get_simulator()
        simulator.reset()

        phos_list = []
        for k in range(K):
            phos_k = simulator(stim_b[k:k+1]).unsqueeze(1)
            phos_list.append(phos_k)

        phos = torch.cat(phos_list, dim=0)   # [K, ...]
        reconstruction = models['decoder'](phos)
        if not use_VLM:
            label_input = chr(sample_indices[b] + ord('A'))
        else:
            label_input = patient.vlm_identify(x[b])
        print(f"label: {label_input}")
        label_inputs.append(label_input)
         
        plot_phos_batch(phos)
        shift_rank_h = input_rank_for_sample(b, K, name="shift")
        rotate_rank_h = input_rank_for_sample(b, K, name="rotate")
        brightness_rank_h = input_rank_for_sample(b, K, name="brightness")
        # rbf_rank_h = input_rank_for_sample(b, K, name="RBF")


        shift_rank_b, shift_event = flip_rank_if_noisy(shift_rank_h, noise_prob, noise_rng)
        rotate_rank_b, rotate_event = flip_rank_if_noisy(rotate_rank_h, noise_prob, noise_rng)
        brightness_rank_b, brightness_event = flip_rank_if_noisy(brightness_rank_h, noise_prob, noise_rng)

        human_shift_rank.append(shift_rank_h)
        human_rotate_rank.append(rotate_rank_h)
        human_brightness_rank.append(brightness_rank_h)

        shift_rank.append(shift_rank_b)
        rotate_rank.append(rotate_rank_b)
        brightness_rank.append(brightness_rank_b)
        # rbf_rank.append(rbf_rank_b)

        noise_events["shift"].append(shift_event)
        noise_events["rotate"].append(rotate_event)
        noise_events["brightness"].append(brightness_event)
        print(f"Noise events({noise_level/100:0.2f}): "
              f"shift: {shift_event['flipped']} {shift_event['u']:.2f}, "
              f"rotate: {rotate_event['flipped']} {rotate_event['u']:.2f}, "
              f"brightness: {brightness_event['flipped']} {brightness_event['u']:.2f}")

    a_w_shift, a_l_shift = build_winner_loser_actions(shift_rank,actions,verbose=True,)
    a_w_rotate, a_l_rotate = build_winner_loser_actions(rotate_rank,actions,verbose=True,)
    a_w_bright, a_l_bright = build_winner_loser_actions(brightness_rank,actions,verbose=True)
    # a_w_rbf, a_l_rbf = build_winner_loser_actions(rbf_rank,actions,verbose=True,)


    dpo_loss_shift = dpo_step(policy=policy,ref_policy=ref_policy,x=x,a_w=a_w_shift,a_l=a_l_shift,beta=beta,
                            optimizer=optimizer_shift,target_params=shift_params,all_param_groups=all_param_groups,)

    dpo_loss_rotate = dpo_step(policy=policy,ref_policy=ref_policy,x=x,a_w=a_w_rotate,a_l=a_l_rotate,beta=beta,
                            optimizer=optimizer_rotate,target_params=rotate_params,all_param_groups=all_param_groups,)
    dpo_loss_bright = dpo_step(policy=policy,ref_policy=ref_policy,x=x,a_w=a_w_bright,a_l=a_l_bright,beta=beta,
                            optimizer=optimizer_brightness,target_params=brightness_params,all_param_groups=all_param_groups,)
    # dpo_loss_rbf = dpo_step(policy=policy,ref_policy=ref_policy,x=x,a_w=a_w_rbf,a_l=a_l_rbf,beta=beta,
    #                           optimizer=optimizer_rbf,target_params=rbf_params,all_param_groups=all_param_groups,)

    
    torch.save(policy.state_dict(), os.path.join(results_dir, f"policy_step_{step:04d}.pt"))

    step_log = {
        "step": step,
        "sample_indices": list(sample_indices),
        "label_inputs": label_inputs,
        "human_ranks": {
            "shift": human_shift_rank,
            "rotate": human_rotate_rank,
            "brightness": human_brightness_rank,
        },
        "used_ranks": {
            "shift": shift_rank,
            "rotate": rotate_rank,
            "brightness": brightness_rank,
        },
        "noise_events": noise_events,
        "losses": {
            "shift": float(dpo_loss_shift.detach().cpu().item()),
            "rotate": float(dpo_loss_rotate.detach().cpu().item()),
            "brightness": float(dpo_loss_bright.detach().cpu().item()),
        },
    }
    logs["steps"].append(step_log)

    if step % 10 == 0 or step == total_steps - 1:
        save_dpo_checkpoint(
            path=os.path.join(results_dir, f"dpo_checkpoint_step_{step:04d}.pt"),
            step=step,
            policy=policy,
            optimizer_shift=optimizer_shift,
            optimizer_rotate=optimizer_rotate,
            optimizer_brightness=optimizer_brightness,
            logs=logs,
            generator=generator,
            noise_rng=noise_rng,
        )


In [ ]:


module_paths = [
    # "base_encoder.affine_backbone",
    # "base_encoder.deform_mean",
    # "base_encoder.deform_log_std",
    "base_encoder.global_shift_mean",
    "base_encoder.global_shift_log_std",
    # "base_encoder.rbf_shift_mean",
    # "base_encoder.rbf_shift_log_std",
    # "base_encoder.rbf_gate_logit",
    # "base_encoder.rbf_raw_sigma",
    "base_encoder.global_rotate_mean",
    "base_encoder.global_rotate_log_std",
]

def get_module(root, path):
    obj = root
    for name in path.split("."):
        obj = getattr(obj, name)
    return obj

for path in module_paths:
    p_mod = get_module(policy, path)
    ref_mod = get_module(ref_policy, path)

    print(f"\n===== {path} =====")
    print("policy:")
    print(p_mod)
    print("ref_policy:")
    print(ref_mod)

In [ ]:

models_group = {
    'new_model': {"encoder": policy, "decoder": models['decoder']},
    'old_model': {"encoder": ref_policy, "decoder": models['decoder']}
}
patients = {
    'new_patient': patient
}

In [ ]:

import visualization
importlib.reload(visualization)
from visualization import *
outputs, sample_indices, figs, axes, eval_answers = visualize_demo_samples_multi_patients(patients=patients,dataset=dataset,dataset_type='calibration_set',models=models_group,deterministic=True,mode="random", cfg=cfg,fig_num=4,)